In [ ]:
import os
import pandas as pd
from collections import defaultdict
import json
from datetime import datetime
import json

data_path = "assets/初赛数据/"
tmp_path = "tmps_data"
output_path = "test_data"

os.makedirs(tmp_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

In [ ]:
# 第一步：合并数据


def merge_csv_files(input_path, out_path):
    # 根据前缀分组文件
    file_groups = defaultdict(list)
    for file_name in os.listdir(input_path):
        if file_name.endswith(".csv") and "字段释义" not in file_name:
            prefix = file_name.rsplit("_", 1)[0]
            file_groups[prefix].append(os.path.join(input_path, file_name))
    # 合并前缀相同的文件
    for prefix, file_list in file_groups.items():
        merged_df = pd.concat(
            (pd.read_csv(file) for file in file_list), ignore_index=True
        )
        output_file = os.path.join(out_path, f"{prefix}.csv")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        merged_df.to_csv(output_file, index=False)
        print(f'合并前缀为"{prefix}"的文件到{output_file}')
    # 将设备参数详情表转为csv
    df_device = pd.read_excel(f"{data_path}设备参数详情.xlsx")
    df_device.to_csv(os.path.join(out_path, "设备参数详情表.csv"), index=False)


merge_csv_files(data_path, tmp_path)
merge_csv_files(data_path, output_path)

In [ ]:
# 第二步：判定A架的开关机和有无电流


def convert_to_numeric(value):
    """
    将值转换为数值类型，无法转换的返回 -1
    """
    try:
        return float(value)
    except ValueError:
        return -1


df = pd.read_csv(os.path.join(tmp_path, "Ajia_plc_1.csv"))
df["Ajia-3_v"] = df["Ajia-3_v"].apply(convert_to_numeric)
df["Ajia-5_v"] = df["Ajia-5_v"].apply(convert_to_numeric)
df["status"] = "False"
df["check_current_presence"] = "False"

for i in range(1, df.shape[0]):
    prev_ajia3 = df.loc[i - 1, "Ajia-3_v"]
    prev_ajia5 = df.loc[i - 1, "Ajia-5_v"]
    curr_ajia3 = df.loc[i, "Ajia-3_v"]
    curr_ajia5 = df.loc[i, "Ajia-5_v"]

    # 停电条件：当前 Ajia-5_v == -1，且前一时刻 Ajia-5_v > 0 或 0
    if curr_ajia5 == -1 and (prev_ajia5 >= 0):
        df.loc[i, "status"] = "停电"

    # A架开机条件：前一时刻 Ajia-3_v == -1，且当前 Ajia-3_v >= 0
    if prev_ajia3 == -1 and curr_ajia3 >= 0:
        df.loc[i, "status"] = "A架开机"
    if prev_ajia5 == -1 and curr_ajia5 >= 0:
        df.loc[i, "status"] = "A架开机"

    # A架关机条件：当前 Ajia-3_v == -1，且前一时刻 Ajia-3_v >= 0
    if curr_ajia3 == -1 and prev_ajia3 >= 0:
        df.loc[i, "status"] = "A架关机"
    if curr_ajia5 == -1 and prev_ajia5 >= 0:
        df.loc[i, "status"] = "A架关机"

    # 有电流条件：前一时刻有一个或全部为0，下一刻均不为0
    if (prev_ajia3 <= 0 or prev_ajia5 <= 0) and (curr_ajia3 > 0 and curr_ajia5 > 0):
        df.loc[i, "check_current_presence"] = "有电流"
    # 无电流条件：前一时刻均不为0，下一刻有一个或全部为0
    elif prev_ajia3 > 0 and prev_ajia5 > 0 and (curr_ajia3 <= 0 or curr_ajia5 <= 0):
        df.loc[i, "check_current_presence"] = "无电流"


# （Ajia-0_v减去Ajia-1_v）的绝对值 ，赋为新列angle_range
def compute_angle_range(row):
    if row["Ajia-0_v"] == "error" or row["Ajia-1_v"] == "error":
        return "error"
    return abs(float(row["Ajia-0_v"]) - float(row["Ajia-1_v"]))


df["angle_range"] = df.apply(compute_angle_range, axis=1)

In [ ]:
# 根据开关机，将A架数据分为若干段
start_time = None
segments = []

for index, row in df.iterrows():
    if row["status"] == "A架开机":
        start_time = row["csvTime"]
    elif row["status"] == "A架关机" and start_time is not None:
        end_time = row["csvTime"]
        segments.append((start_time, end_time))
        start_time = None